# ISL Translator: SOTA Fine-Tuning on ISL-CSLTR

Fine-tunes our Hybrid ST-GCN + Transformer model (pre-trained on the INCLUDE dataset) on the continuous sentence-level ISL-CSLTR dataset.

**Expected inputs:**
1. `isl-translator-codebase` - Latest codebase zip
2. `isl-csltr-processed-keypoints` - Processed ISL-CSLTR dataset
3. Output from INCLUDE pre-training run (contains `best.pt`)

In [ ]:
# --- 1. Dependencies ---
!pip install -q mediapipe==0.10.9 opencv-python transformers sentencepiece tensorboard pyyaml editdistance

In [ ]:
# --- 2. Setup Paths and Files ---
import os, sys, glob, shutil, json
from pathlib import Path

WORKING_DIR = Path("/kaggle/working/data")
CODEBASE_DIR = Path("/kaggle/working/isl-translator")

# Clean working dirs
for d in [WORKING_DIR, CODEBASE_DIR]:
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True)

# --- Find and extract codebase ---
codebase_zip = None
for f in glob.glob("/kaggle/input/**/*.zip", recursive=True):
    if "codebase" in f.lower():
        codebase_zip = f
        break
assert codebase_zip, "Attach isl-translator-codebase as input dataset"
print(f"Extracting codebase: {codebase_zip}")
!unzip -q "{codebase_zip}" -d /kaggle/working/isl-translator
sys.path.insert(0, "/kaggle/working/isl-translator")
os.chdir("/kaggle/working/isl-translator")

# --- Find ISL-CSLTR dataset ---
ann_file = None
for f in glob.glob("/kaggle/input/**/train_annotations.json", recursive=True):
    # Skip any INCLUDE annotations (from pre-training output)
    if "include" in f.lower():
        continue
    ann_file = Path(f)
    break
assert ann_file, "Attach isl-csltr-processed-keypoints as input dataset"

# Figure out the data root (where train/ val/ dirs live)
data_root = ann_file.parent
if not (data_root / "train").exists():
    data_root = ann_file.parent.parent
print(f"ISL-CSLTR data root: {data_root}")

# Copy data to working dir
for subdir in ["train", "val", "test"]:
    src = data_root / subdir
    if src.exists():
        shutil.copytree(str(src), str(WORKING_DIR / subdir))
        print(f"  Copied {subdir}/")

# Copy annotation files
for ann_name in ["train_annotations.json", "val_annotations.json"]:
    for candidate in [data_root / ann_name, ann_file.parent / ann_name]:
        if candidate.exists():
            shutil.copy(str(candidate), str(WORKING_DIR / ann_name))
            print(f"  Copied {ann_name}")
            break

# --- Find pre-trained checkpoint ---
pretrained = None
for name in ["best.pt", "latest.pt"]:
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if hits:
        pretrained = hits[0]
        break
assert pretrained, "Attach the output from INCLUDE pre-training (contains best.pt)"
print(f"Pre-trained weights: {pretrained}")
shutil.copy(pretrained, "/kaggle/working/pretrained.pt")

In [ ]:
# --- 3. Configure Fine-Tuning for SOTA ---
import yaml

with open("configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Data paths
config["paths"]["processed_dir"] = "/kaggle/working/data"
config["paths"]["checkpoint_dir"] = "/kaggle/working/checkpoints"

# Logging
config["logging"]["tensorboard"] = True
config["logging"]["wandb"] = False

# ===== SOTA FINE-TUNING HYPERPARAMETERS =====
# Lower LR for transfer learning (encoder already trained)
config["training"]["learning_rate"] = 0.00003   # 3e-5
config["training"]["min_lr"] = 0.000001          # 1e-6
config["training"]["epochs"] = 200               # Long training at low LR
config["training"]["warmup_epochs"] = 15         # Slow warmup preserves encoder features
config["training"]["label_smoothing"] = 0.0      # Disable for cleaner CTC gradients
config["training"]["gradient_clip"] = 1.0        # Tight clipping
config["training"]["gradient_accumulation"] = 4  # Effective batch = 32
config["training"]["batch_size"] = 8

# Patience 25: Let the model converge fully
config["training"]["early_stopping"] = {
    "patience": 25,
    "min_delta": 0.001,
    "monitor": "val_wer"
}

with open("configs/finetune_config.yaml", "w") as f:
    yaml.dump(config, f)

print("Fine-tuning configuration:")
t = config["training"]
print(f"  LR: {t['learning_rate']} -> {t['min_lr']} (cosine)")
print(f"  Epochs: {t['epochs']}, Warmup: {t['warmup_epochs']}")
print(f"  Effective batch: {t['batch_size']} x {t['gradient_accumulation']} = {t['batch_size'] * t['gradient_accumulation']}")
print(f"  Grad clip: {t['gradient_clip']}, Label smoothing: {t['label_smoothing']}")
print(f"  Early stopping patience: {t['early_stopping']['patience']}")

In [ ]:
# --- 4. Launch SOTA Fine-Tuning ---
!python scripts/train.py \
    --config configs/finetune_config.yaml \
    --finetune /kaggle/working/pretrained.pt \
    --output_dir /kaggle/working/checkpoints

In [ ]:
# --- 5. Results Summary ---
import torch

ckpt_path = "/kaggle/working/checkpoints/best.pt"
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    print(f"Best epoch: {ckpt['epoch']}")
    print(f"Best WER:   {ckpt['best_wer']:.2%}")
else:
    print("No best checkpoint found - check training logs above")